# 01b – Split into test set and cross-validation folds (task 16)

We split the buildings once, and everyone uses this split. Two rules:
- **Groups stay together:** buildings on the same boiler, or twins, have the same value, so they
  must be on the same side. Otherwise a model could predict a building by copying its twin.
- **Houses in their normal share:** houses are only about 9% of the data, so we balance them
  across all parts (feedback point 1).

About 14% of buildings form the test set, used once at the end (task 26). The rest is split into 5 folds.
Output: `data/processed/split.csv` with one row per building: EGID, test (True/False), fold (0–4, empty for test).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

processed = Path("../data/processed")

SEED = 42          # same split every run
TEST_PARTS = 7     # 1 of 7 parts = about 14% test set
N_FOLDS = 5        # folds for cross-validation
THRESHOLD = 450

data = pd.read_csv(processed / "building_clean.csv")
print("Buildings:", len(data), "| houses:", data["house"].sum(), "| groups:", data["group_id"].nunique())

## 1. Test set

`StratifiedGroupKFold` splits the data into 7 parts, keeping groups together ("Group") and
the share of houses about equal in every part ("Stratified"). We take one part as the test set.
`shuffle=True` is needed, otherwise the seed has no effect.

In [ ]:
test_splitter = StratifiedGroupKFold(n_splits=TEST_PARTS, shuffle=True, random_state=SEED)
rest_idx, test_idx = next(test_splitter.split(data, y=data["house"], groups=data["group_id"]))  # first of the 7 parts

data["test"] = False
data.loc[data.index[test_idx], "test"] = True
print("Test set:", len(test_idx), "buildings,", data.loc[data["test"], "house"].sum(), "houses")

## 2. Five folds for cross-validation

The remaining buildings are split the same way into 5 folds (0–4). In each round of
cross-validation, one fold is the validation part and the other four are the training part.

In [ ]:
rest = data[data["test"] == False]
data["fold"] = pd.Series(pd.NA, index=data.index, dtype="Int64")  # empty for test buildings

cv_splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, val_idx) in enumerate(cv_splitter.split(rest, y=rest["house"], groups=rest["group_id"])):
    data.loc[rest.index[val_idx], "fold"] = k   # buildings in the k-th validation part get fold k

print("Buildings without a fold (should be the test set):", data["fold"].isna().sum())

## 3. Checks

Every part should have about the same size and house share, and no group may appear in more than one part.
The share of houses above 450 is not balanced on purpose; with about 200 houses per part it varies by a few points.

In [ ]:
data["part"] = data["fold"].astype("string").fillna("test")  # "0"–"4" or "test"

summary = data.groupby("part").agg(
    buildings=("EGID", "size"),
    houses=("house", "sum"),
    groups=("group_id", "nunique"),
)
summary["share of buildings (%)"] = (summary["buildings"] / len(data) * 100).round(1)
summary["house share (%)"] = (summary["houses"] / summary["buildings"] * 100).round(1)
summary["houses above 450 (%)"] = (
    data[data["house"]].groupby("part")["y"].apply(lambda s: (s > THRESHOLD).mean() * 100).round(1)
)
display(summary)

parts_per_group = data.groupby("group_id")["part"].nunique()
print("Groups split across parts:", (parts_per_group > 1).sum())  # must be 0

assert (parts_per_group == 1).all(), "a group appears in more than one part!"
assert data.loc[data["test"], "fold"].isna().all(), "a test building has a fold number!"
assert data.loc[data["test"] == False, "fold"].notna().all(), "a non-test building has no fold!"

## 4. Save

In [ ]:
split = data[["EGID", "test", "fold"]]
split.to_csv(processed / "split.csv", index=False)
print("Saved split.csv:", len(split), "buildings")